In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
 
# Define the range of hyperparameters to test
hidden_sizes = [16, 32, 64]  # Hidden layer sizes
depths = [2, 4, 6, 8]  # Number of hidden layers
learning_rates = [0.0001, 0.0002, 0.0003, 0.001]  # Learning rates
epochs = 500  # Number of epochs
 
# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
 
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
 
# Check the dimensions
print(f"X_train shape: {X_train_scaled.shape}, X_test shape: {X_test_scaled.shape}")
print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")
 
# Parameters
input_size = X_train.shape[1]  # Number of features
output_size = len(np.unique(y_train))  # Number of unique classes
 
# Custom weight initialization function
def initialize_weights(input_size, hidden_size, output_size, num_hidden_layers):
    np.random.seed(42)  # For reproducibility
    weights = {}
    # Initialize weights for hidden layers and output layer
    for i in range(1, num_hidden_layers + 1):
        if i == 1:
            weights[f'W{i}'] = np.random.randn(input_size, hidden_size) * np.sqrt(2. / input_size)
        else:
            weights[f'W{i}'] = np.random.randn(hidden_size, hidden_size) * np.sqrt(2. / hidden_size)
        weights[f'b{i}'] = np.zeros((1, hidden_size))
    weights[f'W{num_hidden_layers + 1}'] = np.random.randn(hidden_size, output_size) * np.sqrt(2. / hidden_size)
    weights[f'b{num_hidden_layers + 1}'] = np.zeros((1, output_size))
    return weights
 
# Softmax function
def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))  # For numerical stability
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)
 
# Forward propagation function
def forward_propagation(X, weights, num_hidden_layers):
    activations = {}
    A = X
    # Forward pass through all hidden layers
    for i in range(1, num_hidden_layers + 1):
        Z = np.dot(A, weights[f'W{i}']) + weights[f'b{i}']
        A = np.maximum(0, Z)  # ReLU activation
        activations[f'Z{i}'] = Z
        activations[f'A{i}'] = A
    # Output layer
    Z_out = np.dot(A, weights[f'W{num_hidden_layers + 1}']) + weights[f'b{num_hidden_layers + 1}']
    A_out = softmax(Z_out)  # Softmax activation for multi-class classification
    activations[f'Z{num_hidden_layers + 1}'] = Z_out
    activations[f'A{num_hidden_layers + 1}'] = A_out
    return activations
 
# Cross-entropy loss function
def compute_loss(y, A_out):
    m = y.shape[0]
    y_onehot = np.zeros((m, output_size))
    y_onehot[np.arange(m), y] = 1  # One-hot encode the labels
    log_probs = -np.log(A_out + 1e-10)  # Adding small epsilon for numerical stability
    loss = (1 / m) * np.sum(np.sum(y_onehot * log_probs, axis=1))  # Calculate loss
    return loss
 
# Backward propagation function
def backward_propagation(X, y, weights, activations, num_hidden_layers):
    gradients = {}
    m = y.shape[0]
    # One-hot encode the labels
    y_onehot = np.zeros((m, output_size))
    y_onehot[np.arange(m), y] = 1
 
    # Output layer gradients
    dZ = activations[f'A{num_hidden_layers + 1}'] - y_onehot
    gradients[f'dW{num_hidden_layers + 1}'] = (1 / m) * np.dot(activations[f'A{num_hidden_layers}'].T, dZ)
    gradients[f'db{num_hidden_layers + 1}'] = (1 / m) * np.sum(dZ, axis=0, keepdims=True)
 
    # Backpropagate through hidden layers
    for i in range(num_hidden_layers, 0, -1):
        dZ = np.dot(dZ, weights[f'W{i + 1}'].T) * (activations[f'A{i}'] > 0)  # ReLU derivative
        gradients[f'dW{i}'] = (1 / m) * np.dot((X if i == 1 else activations[f'A{i - 1}']).T, dZ)
        gradients[f'db{i}'] = (1 / m) * np.sum(dZ, axis=0, keepdims=True)
 
    return gradients
 
# Evaluation function
def evaluate_model(X, y, weights, num_hidden_layers=2):
    activations = forward_propagation(X, weights, num_hidden_layers)
    predictions = np.argmax(activations[f'A{num_hidden_layers + 1}'], axis=1)  # Get predicted classes
    accuracy = np.mean(predictions == y)  # Calculate accuracy
    return accuracy
 
# Training loop with multiple hyperparameter combinations
results = []  # To store results for all combinations
for hidden_size in hidden_sizes:
    for depth in depths:
        for lr in learning_rates:
            weights = initialize_weights(input_size, hidden_size, output_size, depth)
            losses = []  # List to store losses for the current configuration
 
            for epoch in range(epochs):
                activations = forward_propagation(X_train_scaled, weights, depth)
                loss = compute_loss(y_train.to_numpy(), activations[f'A{depth + 1}'])
                losses.append(loss)
                # Backpropagation and weight updates
                gradients = backward_propagation(X_train_scaled, y_train.to_numpy(), weights, activations, depth)
                for i in range(1, depth + 2):
                    weights[f'W{i}'] -= lr * gradients[f'dW{i}']
                    weights[f'b{i}'] -= lr * gradients[f'db{i}']
 
            # Evaluate performance on train and test sets
            train_accuracy = evaluate_model(X_train_scaled, y_train.to_numpy(), weights, depth)
            test_accuracy = evaluate_model(X_test_scaled, y_test.to_numpy(), weights, depth)
            results.append((hidden_size, depth, lr, train_accuracy, test_accuracy))
            print(f"Train Accuracy: {train_accuracy:.4f}, Test Accuracy: {test_accuracy:.4f} for hidden_size={hidden_size}, depth={depth}, learning_rate={lr}")
 
# Sort results by test accuracy in descending order and display top 5
top_5_results = sorted(results, key=lambda x: x[4], reverse=True)[:5]
 
print("\nTop 5 Performing Configurations:")
for result in top_5_results:
    print(f"Hidden Size: {result[0]}, Depth: {result[1]}, Learning Rate: {result[2]}, Train Accuracy: {result[3]:.4f}, Test Accuracy: {result[4]:.4f}")

NameError: name 'X' is not defined